# ScholarRAG
### Hybrid Retrieval-Augmented QA with Citations over Scientific Papers

ScholarRAG answers questions about a corpus of 200 arXiv papers on visual anomaly detection. Every answer must cite the passage it comes from, and the system should decline when the corpus does not contain the answer.

This notebook builds the foundation that everything else is measured against: the corpus, the chunked text, two evaluation question sets, and baseline retrieval scores for dense and keyword search.

**Author:** Moh Hasbi Alfarizi

## Pipeline

```
arXiv API ──► 400 papers (metadata) ──► filter + topic ranking ──► 200 PDFs
                                                                     │
                                  PyMuPDF parsing ──► chunks (~350 words, 60 overlap)
                                                                     │
                          ┌──────────────────────────────────────────┴──────────┐
                  Dense index (bge-small-en-v1.5)                         BM25 index
                          └─────────────────────► retrieval ◄───────────────────┘
                                                      │
                      evaluation on frozen question sets: Hit@k, MRR, PaperHit@10
```

Planned next: hybrid fusion (RRF) and a cross-encoder reranker, then an LLM that answers with citations or refuses.

## Project status

| Milestone | What | Status |
|---|---|---|
| M0 | Corpus collection and question sets | ✅ corpus done · ✅ LLM questions frozen · ✅ handwritten questions frozen |
| M1 | PDF parsing and chunking | ✅ |
| M2 | Baseline retrieval (dense vs BM25) + metrics | ✅ |
| – | Chunk-size experiment (350 / 250 / 180 words) | ✅ keep 350 (`02_chunk_size_experiment.ipynb`) |
| M3 | Hybrid search (RRF) + reranker | ✅ BM25 ∪ dense top 10 → `bge-reranker-base` (`03_hybrid_rerank.ipynb`) |
| M4 | Answers with citations and refusal | ✅ Gemini, verified quotes; 20 / 20 unanswerable refused (`04_answers_citations.ipynb`) |
| M5 | Answer evaluation (citations, refusals, cost, latency) | ✅ acceptable 0.92 (A) / 0.93 (B1), 0 incorrect, 20 / 20 near-miss refused (`05_evaluation.ipynb`) |
| M6 | FastAPI + UI, deployment | 🔧 local API + web UI done (`src/app.py`); deployment pending |

## Results so far

Baseline retrieval on the 64 frozen LLM-generated questions (`eval/questions_v1.csv`, 3,910 indexed chunks):

| Retriever | Hit@1 | Hit@5 | Hit@10 | Hit@20 | MRR | PaperHit@10 |
|---|---|---|---|---|---|---|
| Dense (bge-small-en-v1.5) | 0.391 | 0.656 | 0.797 | 0.828 | 0.512 | 0.922 |
| BM25 | **0.625** | **0.906** | **0.938** | **0.969** | **0.751** | **1.000** |

BM25 wins clearly here, but these questions were written by an LLM from a single passage and tend to reuse its wording, which favours keyword search. The handwritten set (section 7) tests that.

**Per question source** (section 7.5; 15 handwritten answerable questions):

| Source | Retriever | Hit@1 | Hit@10 | MRR |
|---|---|---|---|---|
| LLM (n = 64) | Dense | 0.391 | 0.797 | 0.512 |
| LLM (n = 64) | BM25 | **0.625** | **0.938** | **0.751** |
| Handwritten (n = 15) | Dense | **0.600** | 0.800 | **0.665** |
| Handwritten (n = 15) | BM25 | 0.200 | **0.933** | 0.442 |

On paraphrased questions the picture flips: dense puts the right chunk first far more often, while BM25 still gets it into the top 10 slightly more often. Part of BM25's lead on set A therefore comes from question wording. With 15 questions one question is 6.7 points, so this is a strong hint rather than a final result. It supports combining both retrievers (M3).

**Chunk size** (`02_chunk_size_experiment.ipynb`): 250- and 180-word chunks remove dense truncation (22.5% → 0.5% → 0%) but do not improve dense retrieval, and BM25 on set A gets worse (MRR 0.751 → 0.659). 350 words stays the default; 250 will be retested with the reranker, whose 512-token limit covers question + passage.

## Known limitations

- **Question wording bias.** LLM-generated questions share vocabulary with their source passage, so they favour BM25. Results are reported separately per question source.
- **Dense truncation.** 22.5% of chunks are longer than the 512-token limit of `bge-small-en-v1.5`; the tail of those chunks is not embedded. Shorter chunks remove this but did not improve dense retrieval (see the chunk-size experiment).
- **Lost content.** Tables, equations and very short text blocks are dropped during parsing, so answers that live only in a table cannot be retrieved.
- **Heuristic cleaning.** Bibliography sections are removed by heuristics (section heading + citation-pattern filter); a few reference fragments may remain.
- **Small evaluation set.** With 64 questions, one question ≈ 1.6 percentage points, so differences of 2–3 points are noise. With the 15 handwritten questions, one question ≈ 6.7 points.
- **Partial verification.** The LLM question set was filtered automatically and through failure review, not checked line by line by hand.

## Data layout (Google Drive)

```
MyDrive/scholar-rag/data/
├── papers_meta.jsonl            # 400 arXiv search results (metadata)
├── selected.jsonl               # 200 papers kept after filtering and ranking
├── raw/                         # 200 PDFs
├── chunks.jsonl                 # 3,991 chunks
├── chunks_clean.jsonl           # 3,910 chunks after removing bibliography chunks
├── index/
│   ├── emb_bge_small.npy        # cached dense embeddings
│   └── chunk_ids.json           # chunk order for the cache
└── eval/
    ├── candidates.csv                   # 66 LLM-generated candidate questions
    ├── unanswerable_candidates.csv      # 12 LLM-generated unanswerable questions
    ├── questions_v1.csv                 # 64 frozen questions used for evaluation
    └── questions_handwritten_v1.csv     # 15 answerable + 20 unanswerable (in progress)
```

## How to run this notebook

- Runs on **Google Colab**. A T4 GPU speeds up embedding but is not required.
- API keys live in **Colab Secrets** (`ANTHROPIC_API_KEY`, `GEMINI_API_KEY`), never in code.
- Sections 1–3 write to `/root/Documents/...`, which Colab wipes when the runtime resets. Section 4 copies everything to Google Drive.
- **Fresh session with data already on Drive:** run section 0, then start at **section 6** (it reloads everything from Drive). Do not re-run section 5.4: LLM output is not deterministic and would overwrite the question candidates.

---
## 0. Environment setup

Installs the libraries used across the notebook: `arxiv` (search API client), `pymupdf` (PDF text extraction), `sentence-transformers` (dense embeddings), `rank_bm25` (keyword search), and the LLM SDKs.

In [1]:
# Lokal: library sudah terpasang di .venv (lihat requirements.txt).
# Jika perlu install ulang, jalankan di terminal:
#   uv pip install --python .venv/bin/python -r requirements.txt


---
## 1. Corpus collection

Goal: a corpus that is large enough to make retrieval non-trivial, but narrow enough that I can judge whether answers are correct. The topic is **visual anomaly detection**, with a focus on industrial images.

### 1.1 Fetch metadata from arXiv

Searches arXiv for computer-vision papers (`cat:cs.CV`) whose abstract mentions "anomaly detection" and stores the first 400 results by relevance. Only metadata is downloaded here (id, title, abstract, date, PDF link); PDFs come later, after filtering.

The client waits 3 seconds between pages, as arXiv asks API users to do.

In [2]:
# src/fetch_metadata.py
import arxiv, json, os, time
from pathlib import Path

BASE = Path.home() / "Projects" / "scholar-rag"
(BASE / "data" / "raw").mkdir(parents=True, exist_ok=True)
(BASE / "data" / "eval").mkdir(parents=True, exist_ok=True)
meta_path = BASE / "data" / "papers_meta.jsonl"

FORCE_REFETCH = False   # True = ambil ulang dari arXiv walaupun file sudah ada

if meta_path.exists() and not FORCE_REFETCH:
    n = sum(1 for _ in open(meta_path))
    print(f"papers_meta.jsonl sudah ada ({n} paper), fetch arXiv dilewati.")
else:
    client = arxiv.Client(page_size=100, delay_seconds=3, num_retries=3)
    search = arxiv.Search(
        query='cat:cs.CV AND abs:"anomaly detection"',
        max_results=400,
        sort_by=arxiv.SortCriterion.Relevance,
    )

    # arXiv kadang membalas HTTP 429 (terlalu banyak request): tunggu lalu coba lagi
    for attempt in range(1, 6):
        try:
            papers = []
            for r in client.results(search):
                papers.append({
                    "id": r.get_short_id(),
                    "title": r.title,
                    "abstract": r.summary,
                    "published": r.published.isoformat(),
                    "pdf_url": r.pdf_url,
                })
            break
        except arxiv.HTTPError as e:
            wait = 60 * attempt
            print(f"percobaan {attempt}: {e} -> tunggu {wait} detik")
            time.sleep(wait)
    else:
        raise RuntimeError("arXiv masih menolak (429). Coba lagi nanti.")

    with open(meta_path, "w") as f:
        for p in papers:
            f.write(json.dumps(p) + "\n")
    print(len(papers), "paper disimpan di", meta_path)


HTTPError: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=cat%3Acs.CV+AND+abs%3A%22anomaly+detection%22&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100)

### 1.2 Reload the metadata

Sanity check that the file exists and loads into a DataFrame (400 rows). `id` is read as a string so arXiv ids like `2301.01234` are not parsed as numbers.

In [1]:
import pandas as pd
from pathlib import Path

BASE = Path.home() / "Projects" / "scholar-rag" / "data"
print(BASE.exists(), (BASE / "papers_meta.jsonl").exists())   # keduanya harus True

df = pd.read_json(BASE / "papers_meta.jsonl", lines=True, dtype={"id": str})
print(len(df))

True True
400


### 1.3 Filter and rank by topic relevance

Reading 400 abstracts by hand is slow, so ranking is automated:

1. **Filter:** drop duplicate titles, very short abstracts (< 300 characters) and papers before 2021 → **340 papers**.
2. **Rank:** embed each `title + abstract` and a one-sentence topic description with `bge-small-en-v1.5`, then score each paper by cosine similarity to the topic. Embeddings are L2-normalised, so the dot product equals cosine similarity.

The printout shows the top 10, a borderline band (ranks 150–170) and the bottom 10. The top is clearly on topic (industrial image anomaly detection); the bottom is mostly medical imaging, plus radar and audio, which confirms the ranking works. The embedding matrix is re-ordered together with the DataFrame so row *i* always matches vector *i*.

In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Tahap 1: saring yang jelas tidak cocok
df["year"] = pd.to_datetime(df["published"]).dt.year
df["title_norm"] = df["title"].str.lower().str.replace(r"\W+", " ", regex=True)
df = df.drop_duplicates("title_norm")
df = df[df["abstract"].str.len() > 300]
df = df[df["year"] >= 2021]          # sesuaikan
df = df.reset_index(drop=True)
print(len(df), "paper tersisa")

# Tahap 2: ranking dengan embedding
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
TOPIC = "unsupervised anomaly detection for industrial images using feature embeddings"

texts = (df["title"] + ". " + df["abstract"]).tolist()
emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
q = model.encode([TOPIC], normalize_embeddings=True)

df["score"] = (emb @ q.T).ravel()
order = np.argsort(-df["score"].values)      # urutan dari skor tertinggi
df = df.iloc[order].reset_index(drop=True)
emb = emb[order]                             # embedding ikut diurutkan, tetap sinkron

pd.set_option("display.max_colwidth", 90)
print(df[["score", "title"]].head(10))
print(df[["score", "title"]].iloc[150:170])
print(df[["score", "title"]].tail(10))

W1017 01:06:31.386000 70238 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


340 paper tersisa


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

      score  \
0  0.893232   
1  0.871348   
2  0.861538   
3  0.858011   
4  0.855237   
5  0.854185   
6  0.852926   
7  0.848730   
8  0.848232   
9  0.844280   

                                                                                       title  
0                A Survey on Unsupervised Anomaly Detection Algorithms for Industrial Images  
1  SLSG: Industrial Image Anomaly Detection by Learning Better Feature Embeddings and One...  
2                                          Deep Industrial Image Anomaly Detection: A Survey  
3  AnomalousPatchCore: Exploring the Use of Anomalous Samples in Industrial Anomaly Detec...  
4         MMVIAD: Multi-view Multi-task Video Understanding for Industrial Anomaly Detection  
5                       UniADC: A Unified Framework for Anomaly Detection and Classification  
6  Component-aware anomaly detection framework for adjustable and logical industrial visu...  
7  Anomaly Detection for Industrial Applications, Its Challenges, Solution

### 1.4 Check topical diversity with clustering

K-means (k = 5) on the embeddings of the top 200 papers, to see which sub-topics the corpus covers. The number after each cluster is its full size; only the four highest-ranked titles are printed.

| Cluster | Theme (from titles) | Papers |
|---|---|---|
| 0 | Video anomaly detection | 14 |
| 1 | Surveys and general industrial image AD | 66 |
| 2 | Multimodal (RGB + 3D) industrial AD | 23 |
| 3 | Unified, zero/few-shot and benchmark papers | 44 |
| 4 | Feature-embedding and autoencoder methods | 53 |

The corpus is diverse, which makes for varied evaluation questions. The video cluster is slightly off the industrial focus but was kept.

In [3]:
from sklearn.cluster import KMeans

top = df.head(200).copy()
top["cluster"] = KMeans(n_clusters=5, random_state=0, n_init=10).fit_predict(emb[:200])

for c in sorted(top["cluster"].unique()):
    print(f"\n=== Cluster {c} ({(top.cluster == c).sum()} paper) ===")
    for t in top[top.cluster == c]["title"].head(4):
        print(" -", t)


=== Cluster 0 (14 paper) ===
 - Vision-Language Models Assisted Unsupervised Video Anomaly Detection
 - Abnormal Event Detection In Videos Using Deep Embedding
 - Generative Cooperative Learning for Unsupervised Video Anomaly Detection
 - Synthetic Pseudo Anomalies for Unsupervised Video Anomaly Detection: A Simple yet Efficient Framework based on Masked Autoencoder

=== Cluster 1 (66 paper) ===
 - A Survey on Unsupervised Anomaly Detection Algorithms for Industrial Images
 - Deep Industrial Image Anomaly Detection: A Survey
 - AnomalousPatchCore: Exploring the Use of Anomalous Samples in Industrial Anomaly Detection
 - MMVIAD: Multi-view Multi-task Video Understanding for Industrial Anomaly Detection

=== Cluster 2 (23 paper) ===
 - BridgeNet: A Unified Multimodal Framework for Bridging 2D and 3D Industrial Anomaly Detection
 - Multimodal Industrial Anomaly Detection via Geometric Prior
 - Multimodal Industrial Anomaly Detection via Hybrid Fusion
 - M3DM-NR: RGB-3D Noisy-Resistant In

### 1.5 Limit survey papers

Surveys are long and mention almost every method by name, so they can crowd out the original papers in retrieval. The rule keeps at most 5 surveys; only 4 were found, so all were kept.

In [4]:
top["is_survey"] = top["title"].str.contains("survey|review|overview", case=False)
print(top["is_survey"].sum(), "paper survei")

4 paper survei


### 1.6 Save the selected corpus

Writes the final 200 papers to `selected.jsonl`. This file is the single source of truth for which papers are in the corpus.

In [5]:
selected = top[~top["is_survey"] | (top.groupby("is_survey").cumcount() < 5)]
selected = selected.head(200)
selected[["id", "title", "abstract", "pdf_url", "published", "score"]].to_json(
    BASE / "selected.jsonl", orient="records", lines=True, force_ascii=False
)
print(len(selected))

200


### 1.7 Download the PDFs

Design choices that make the download safe to re-run:

- **Resumable:** a PDF that already exists is skipped (`skip`), so an interrupted run continues where it stopped.
- **Validated:** a file is only saved if it starts with the `%PDF` header.
- **Polite:** 3-second pause between requests.

All 200 downloads succeeded.

In [6]:
import requests, time

(BASE / "raw").mkdir(parents=True, exist_ok=True)

def download(row):
    out = BASE / "raw" / (row["id"].replace("/", "_") + ".pdf")
    if out.exists():
        return "skip"
    r = requests.get(row["pdf_url"], timeout=60,
                     headers={"User-Agent": "scholar-rag-student-project"})
    if r.ok and r.content[:4] == b"%PDF":
        out.write_bytes(r.content)
        time.sleep(3)
        return "ok"
    time.sleep(3)
    return f"gagal ({r.status_code})"

for _, row in selected.iterrows():   # uji dengan 20 dulu
    print(row["id"], download(row))

2204.11161v4 ok


2305.00398v1 ok


2301.11514v5 ok


2408.15113v2 ok


2605.10833v1 ok


...

2403.09303v3 ok


2403.14213v1 ok


2505.03412v1 ok


2310.07716v1 ok


2409.05378v1 ok


---
## 2. PDF quality check

A scanned or broken PDF has little or no extractable text and would silently poison retrieval. The check flags any PDF with fewer than 800 characters per page. **None were flagged.**

In [7]:
# Lokal: pymupdf sudah terpasang di .venv


In [8]:
import pymupdf

bad = []
for pdf in sorted((BASE / "raw").glob("*.pdf")):
    doc = pymupdf.open(pdf)
    chars = sum(len(p.get_text()) for p in doc)
    per_page = chars / max(len(doc), 1)
    if per_page < 800:
        bad.append((pdf.name, len(doc), round(per_page)))
print("Mencurigakan:", bad)

Mencurigakan: []


Spot check of raw extracted text (page 3 of one paper). Figure labels come out as short fragments ("Generator (G)", "V1", …) while body text is clean. The chunker in section 3 drops those short fragments.

In [9]:
doc = pymupdf.open(next((BASE / "raw").glob("*.pdf")))
print(doc[2].get_text()[:1500])

CRAD
3
leading to improved generalization performance. However, such strength may
turn into a drawback when testing abnormal inputs. If these abnormal inputs
can be reconstructed using a combination of normal features, the model could
potentially generate outputs that are identical to the abnormal inputs. This is-
sue, referred to as an identity shortcut (IS) by UniAD [37], prevents the models
from detecting anomalies due to the minimal difference between the abnormal
input and produced output.
Furthermore, a significant limitation exists in most of the approaches dis-
cussed earlier, as they are primarily designed to handle only one class of objects
per model. When these methods are extended to multi-class scenarios, where a
single model handles multiple classes, a significant performance drop has been
observed [37], even with state-of-the-art methods. To mitigate this limitation,
memory-based methods may incorporate sufficient memory to accommodate
multi-class normality, yet this sim

---
## 3. Parsing and chunking

Each PDF is split into **chunks**: passages small enough to retrieve precisely, large enough to carry context. Every chunk keeps its paper id, title and page range, which is what makes citations possible later.

| Decision | Value | Reason |
|---|---|---|
| Chunk size | ~350 words | Enough context for one idea; small enough for precise retrieval |
| Overlap | 60 words | Longer than any evidence quote (≤ 25 words), so every quote sits fully inside at least one chunk |
| Boundaries | Cut at a paragraph break in the last 25% of the window when possible | Avoids splitting sentences |
| References | Everything after a "References"/"Bibliography" heading is dropped (heading must appear after the first 40% of the document) | Bibliographies are full of method names and pollute keyword search |
| Noise | Blocks under 40 characters or under 50% letters are dropped | Page numbers, figure labels, equations, tables |
| Hyphenation | Words split across lines are re-joined | Keeps tokens intact for BM25 |

**Result:** 3,991 chunks from 200 papers, no failures.

In [10]:
import re, json, random
from pathlib import Path
import fitz  # pymupdf

BASE = Path.home() / "Projects" / "scholar-rag" / "data"   # sesuaikan jika perlu

# peta id -> judul, dari selected.jsonl
meta = {}
for line in open(BASE / "selected.jsonl"):
    r = json.loads(line)
    meta[r["id"].replace("/", "_")] = r["title"]

REF_RE = re.compile(r"^\s*(\d+\.?\s*)?(references|bibliography)\s*$", re.I)

def extract_units(pdf_path):
    """Kembalikan list (page, paragraf) dengan References sudah dibuang."""
    doc = fitz.open(pdf_path)
    units = []
    ref_at = None   # index di units tempat References mulai
    for pno, page in enumerate(doc, start=1):
        for b in page.get_text("blocks"):
            if b[6] != 0:               # 6 = tipe blok; 0 = teks, 1 = gambar
                continue
            t = b[4]
            t = re.sub(r"-\n(\w)", r"\1", t)          # sambung kata yang terpotong tanda hubung
            t = re.sub(r"\s+", " ", t).strip()
            if len(t) < 40:                            # buang nomor halaman, label kecil
                if REF_RE.match(t) and pno > len(doc) * 0.4:
                    ref_at = len(units)                # mulai dari sini sampai akhir dibuang
                continue
            letters = sum(ch.isalpha() for ch in t)
            if letters / len(t) < 0.5:                 # blok yang kebanyakan simbol/rumus
                continue
            units.append((pno, t))
    return units[:ref_at] if ref_at is not None else units

def chunk_units(units, target=350, overlap=60):
    words, pages, boundaries = [], [], set()
    for pno, t in units:
        boundaries.add(len(words))        # indeks kata tempat paragraf baru dimulai
        w = t.split()
        words += w
        pages += [pno] * len(w)
    n = len(words)
    chunks, start = [], 0
    while start < n:
        end = min(start + target, n)
        if end < n:                       # coba potong di batas paragraf di 25% terakhir window
            cands = [b for b in boundaries if start + int(0.75 * target) < b <= start + target]
            if cands:
                end = max(cands)
        chunks.append((" ".join(words[start:end]), pages[start], pages[end - 1]))
        if end >= n:
            break
        start = max(end - overlap, start + 1)
    return chunks

out_path = BASE / "chunks.jsonl"
n_chunks, failed = 0, []
with open(out_path, "w") as f:
    for pdf in sorted((BASE / "raw").glob("*.pdf")):
        pid = pdf.stem
        try:
            units = extract_units(pdf)
            for i, (text, p0, p1) in enumerate(chunk_units(units)):
                f.write(json.dumps({
                    "chunk_id": f"{pid}_c{i:04d}",
                    "paper_id": pid,
                    "title": meta.get(pid, ""),
                    "page": p0,
                    "page_end": p1,
                    "text": text,
                    "n_words": len(text.split()),
                }, ensure_ascii=False) + "\n")
                n_chunks += 1
        except Exception as e:
            failed.append((pid, str(e)))

print("Total chunk:", n_chunks, "| gagal:", failed)

Total chunk: 3991 | gagal: []


### 3.1 Chunk statistics

- Words per chunk: median 332, max 350. Many chunks end slightly below 350 because the cut moves to a paragraph break.
- Chunks per paper: 6 to 48, median 19. No paper is empty.

In [11]:
import pandas as pd
ch = pd.read_json(out_path, lines=True, dtype={"paper_id": str})
print(ch["n_words"].describe())
print(ch.groupby("paper_id").size().describe())     # chunk per paper
print(ch.groupby("paper_id").size().sort_values().head(5))   # paper dengan chunk paling sedikit

count    3991.000000
mean      319.586570
std        38.630535
min        65.000000
25%       305.000000
50%       332.000000
75%       347.000000
max       350.000000
Name: n_words, dtype: float64
count    200.000000
mean      19.955000
std        7.530372
min        6.000000
25%       15.000000
50%       19.000000
75%       23.000000
max       48.000000
dtype: float64
paper_id
2512.18219v1    6
2310.16435v1    8
2509.05034v1    8
2202.08341v1    8
2104.04289v1    9
dtype: int64


### 3.2 Read random chunks

Statistics can hide parsing problems, so eight random chunks are read by eye. Most read cleanly. One of them (`2508.05503v1_c0018`) is a **bibliography fragment** that slipped past the heading rule; this led to the extra citation-pattern filter in section 6.2.

In [12]:
for _, r in ch.sample(8, random_state=1).iterrows():
    print(f"--- {r.chunk_id} | hal. {r.page}-{r.page_end} | {r.n_words} kata")
    print(r.title)
    print(r.text[:700], "...\n")

--- 2508.05503v1_c0018 | hal. 9-10 | 350 kata
AutoIAD: Manager-Driven Multi-Agent Collaboration for Automated Industrial Anomaly Detection
B.; Zheng, B.; Yu, B.; Li, C.; Liu, D.; Huang, F.; Wei, H.; Lin, H.; Yang, J.; Tu, J.; Zhang, J.; Yang, J.; Yang, J.; Zhou, J.; Lin, J.; Dang, K.; Lu, K.; Bao, K.; Yang, K.; Yu, L.; Li, M.; Xue, M.; Zhang, P.; Zhu, Q.; Men, R.; Lin, R.; Li, T.; Xia, T.; Ren, X.; Ren, X.; Fan, Y.; Su, Y.; Zhang, Y.; Wan, Y.; Liu, Y.; Cui, Z.; Zhang, Z.; and Qiu, Z. 2024a. Qwen2.5 Technical Report. arXiv preprint arXiv:2412.15115. Yang, Z.; Zeng, W.; Jin, S.; Qian, C.; Luo, P.; and Liu, W. 2024b. AutoMMLab: Automatically Generating Deployable Models from Language Instructions for Computer Vision Tasks. CoRR, abs/2402.15351. Yu, J.; Zheng, Y.; Wang, X.; Li, W.; Wu, Y.; Zhao, R.; and Wu, L. 2021. FastFlow: Unsup ...

--- 2511.06644v3_c0006 | hal. 6-8 | 346 kata
UniADC: A Unified Framework for Anomaly Detection and Classification
for more challenging anomaly clas- Traini

### 3.3 Residual reference check

Share of chunks that match a citation pattern such as `[12] Smith, …`. The result, **3.5%**, is low; some of these are ordinary sentences that cite a paper.

In [13]:
print(ch["text"].str.contains(r"\[\d+\]\s+[A-Z][a-z]+,?\s", regex=True).mean())

0.03507892758707091


### 3.4 Papers with the fewest chunks

A paper with very few chunks could mean failed extraction. The five smallest are all short papers (4–6 pages, 6–9 chunks), so the counts are expected.

In [14]:
import fitz, pandas as pd

chunks = pd.read_json(BASE / "chunks.jsonl", lines=True, dtype={"paper_id": str})
small = chunks.groupby("paper_id").size().sort_values().head(5).index
for pid in small:
    pages = len(fitz.open(BASE / "raw" / f"{pid}.pdf"))
    n = (chunks.paper_id == pid).sum()
    print(pid, "|", pages, "halaman |", n, "chunk |", meta[pid][:70])

2512.18219v1 | 4 halaman | 6 chunk | Unsupervised Anomaly Detection with an Enhanced Teacher for Student-Te
2310.16435v1 | 5 halaman | 8 chunk | On Pixel-level Performance Assessment in Anomaly Detection
2509.05034v1 | 6 halaman | 8 chunk | Towards Efficient Pixel Labeling for Industrial Anomaly Detection and 
2202.08341v1 | 5 halaman | 8 chunk | Anomalib: A Deep Learning Library for Anomaly Detection
2104.04289v1 | 5 halaman | 9 chunk | MLF-SC: Incorporating multi-layer features to sparse coding for anomal


---
## 4. Persisting data on Google Drive

Colab's local disk is wiped whenever the runtime resets; Google Drive is not. From this point on, `BASE` points to Drive and every file is read from and written there.

1. Check where `BASE` currently points and whether `chunks.jsonl` exists.
2. Mount Drive and point `BASE` to `MyDrive/scholar-rag/data`.
3. Copy the local data (metadata, PDFs, chunks) to Drive. Only needed once, right after sections 1–3.

In [15]:
from pathlib import Path
print(BASE, "|", (BASE / "chunks.jsonl").exists())

/Users/mhasbiarz/Projects/scholar-rag/data | True


In [16]:
# Lokal: tidak perlu mount Google Drive. BASE menunjuk ke folder data di proyek ini.
from pathlib import Path
BASE = Path.home() / "Projects" / "scholar-rag" / "data"
BASE.mkdir(parents=True, exist_ok=True)
print(BASE)


/Users/mhasbiarz/Projects/scholar-rag/data


In [17]:
# Lokal: langkah salin ke Drive tidak diperlukan (data sudah di BASE).


---
## 5. Evaluation set A: LLM-generated questions

Retrieval can only be measured with questions whose correct passage is known in advance (the **gold** passage). Writing hundreds of these by hand is slow, so an LLM drafts them:

1. Sample **one chunk per paper** (100 chunks), skipping short chunks and table-heavy chunks (> 12% digits).
2. For each chunk, the LLM writes one question that only that passage answers, a short answer, and an **exact evidence quote** (8–25 words).
3. A candidate is rejected if the quote is not found verbatim in the passage, which filters out hallucinated evidence.

**Why gold is stored as `paper_id` + `evidence`, not `chunk_id`:** chunk ids change whenever the chunking changes. A quote does not, so the same question set can evaluate any chunking scheme.

The prompt forces questions to name the method, dataset or setting. With 200 papers on one topic, a generic question ("Which dataset is used?") would match dozens of papers and make the gold answer meaningless.

### 5.1 Sample source chunks

Fixed `random_state` values make the sample identical in every session.

In [21]:
import pandas as pd

chunks = pd.read_json(BASE / "chunks.jsonl", lines=True, dtype={"paper_id": str})

pool = chunks[chunks.n_words >= 250].copy()
pool["digit_ratio"] = pool.text.apply(lambda t: sum(c.isdigit() for c in t) / len(t))
pool = pool[pool.digit_ratio < 0.12]
cands = pool.groupby("paper_id").sample(n=1, random_state=0).sample(100, random_state=0)

print(len(chunks), "chunk |", len(cands), "kandidat chunk")

3991 chunk | 100 kandidat chunk


### 5.2 Option A: Anthropic client (set up, not used for the final run)

Loads the Anthropic key from Colab Secrets and creates a client. The final questions were generated with Gemini (5.3–5.4), so this cell is kept only for reference.

In [17]:
from dotenv import load_dotenv
load_dotenv()   # baca API key dari file .env
import os, anthropic

key = os.getenv("ANTHROPIC_API_KEY")
print("key termuat:", bool(key), "| awalan:", key[:7] if key else "-")   # harus True dan 'sk-ant-'

# opsional: hanya dibuat kalau key ada (tidak dipakai di run final)
client = anthropic.Anthropic(api_key=key) if key else None


key termuat: True | awalan: sk-ant-


### 5.3 Option B: Gemini client and smoke test

Installs the Google GenAI SDK and sends one tiny request to confirm the key and model work before running 100 calls.

> **Security note:** an earlier version of this cell contained the API key in plain text. It now reads the key from Colab Secrets. A key that was ever written in a notebook should be revoked and replaced.

In [22]:
# Lokal: google-genai sudah terpasang di .venv


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 19.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


In [23]:
import os
from dotenv import load_dotenv
load_dotenv()   # baca API key dari file .env
from google import genai

GEMINI_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_KEY:
    print("GEMINI_API_KEY kosong di .env, smoke test dilewati.")
else:
    client = genai.Client(api_key=GEMINI_KEY)
    try:
        resp = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents="Explain how AI works in a few words",
        )
        print(resp.text)
    except Exception as e:
        print("Smoke test gagal:", e)


AI finds patterns in massive amounts of data to make predictions, decisions, or creations.


### 5.4 Generate candidate questions

Runs the generator over the 100 sampled chunks with `temperature = 0`, waiting 4.5 s between calls to stay under the free-tier rate limit (15 requests/minute).

**Result:** 66 candidates kept out of 100. Of the 34 rejected, 18 were `503 UNAVAILABLE` (model overloaded) rather than quality rejections, so the yield could be raised by retrying only those.

> ⚠️ **Do not re-run this cell.** LLM output is not deterministic: an earlier run produced 71 candidates, this one 66. The evaluated set is frozen in section 6.8.

In [24]:
import os
import json, re, random, time
import pandas as pd
from google import genai
from google.genai import types
from dotenv import load_dotenv
load_dotenv()   # baca API key dari file .env

# Initialize working Gemini client
GEMINI_KEY = os.getenv('GEMINI_API_KEY')

SYSTEM = """You write evaluation questions for a QA system over ~200 research papers on anomaly detection.
Given ONE passage from a paper (plus its title), write ONE question that only this passage can answer.
Rules:
- The question must be self-contained and specific enough that only this paper fits: name the method, dataset or setting. Never write "this paper", "the authors" or "the passage".
- Paraphrase; do not copy any phrase of 6+ words from the passage.
- Prefer methods, design choices, results, or limitations.
- If the passage has no substantive content (tables of numbers, acknowledgements, boilerplate), return {"skip": true}.
- answer: 1-2 sentences.
- evidence: an EXACT quote copied verbatim from the passage, 8-25 words, supporting the answer.
- type: one of "factual", "method", "limitation".
Return ONLY JSON: {"question","answer","evidence","type"} or {"skip": true}."""

def norm(s):
    return re.sub(r"\s+", " ", s).strip().lower()

def gen(text, title):
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM,
        max_output_tokens=2048,
        temperature=0.0
    )
    try:
        msg = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=f"Paper title: {title}\n\nPassage:\n{text}",
            config=config
        )

        raw = msg.text.strip()
        raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.M).strip()

        d = json.loads(raw)
        if d.get("skip"):
            return None
        if norm(d["evidence"]) not in norm(text):
            return None
        return d
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

# # select candidates single test
# pool = chunks[chunks.n_words >= 250].copy()
# pool["digit_ratio"] = pool.text.apply(lambda t: sum(c.isdigit() for c in t) / len(t))
# pool = pool[pool.digit_ratio < 0.12]
# cands = pool.groupby("paper_id").sample(n=1, random_state=0).sample(100, random_state=0)

# r = cands.iloc[0]
# print(r.title)
# print(gen(r.text, r.title))

# # pilih kandidat: chunk cukup panjang, bukan tabel angka, 1 per paper
# pool = chunks[chunks.n_words >= 250].copy()
# pool["digit_ratio"] = pool.text.apply(lambda t: sum(c.isdigit() for c in t) / len(t))
# pool = pool[pool.digit_ratio < 0.12]
# cands = pool.groupby("paper_id").sample(n=1, random_state=0).sample(100, random_state=0)

# rows = []
# for _, r in cands.iterrows():
#     try:
#         d = gen(r.text, r.title)
#     except Exception as e:
#         print("skip:", r.paper_id, e); continue
#     if d:
#         rows.append({"id": f"q{len(rows)+1:03d}", "keep": "", **d,
#                      "paper_id": r.paper_id, "page": r.page, "page_end": r.page_end,
#                      "answerable": True, "source": "llm"})

# (BASE / "eval").mkdir(exist_ok=True)
# pd.DataFrame(rows).to_csv(BASE / "eval" / "candidates.csv", index=False)
# print(len(rows), "kandidat")

cand_path = BASE / "eval" / "candidates.csv"
if cand_path.exists():
    print("candidates.csv sudah ada, generate dilewati (output LLM tidak deterministik).")
elif not GEMINI_KEY:
    print("GEMINI_API_KEY kosong di .env, generate dilewati.")
else:
    client = genai.Client(api_key=GEMINI_KEY)
    # Pilih kandidat: chunk cukup panjang, bukan tabel angka, 1 per paper
    pool = chunks[chunks.n_words >= 250].copy()
    pool["digit_ratio"] = pool.text.apply(lambda t: sum(c.isdigit() for c in t) / len(t))
    pool = pool[pool.digit_ratio < 0.12]
    cands = pool.groupby("paper_id").sample(n=1, random_state=0).sample(100, random_state=0)

    rows = []
    for idx, (_, r) in enumerate(cands.iterrows()):
        print(f"Processing {idx+1}/100: Paper ID {r.paper_id}...", end=" ")

        try:
            d = gen(r.text, r.title)
        except Exception as e:
            print("skip akibat exception luar:", r.paper_id, e)
            continue

        if d:
            rows.append({"id": f"q{len(rows)+1:03d}", "keep": "", **d,
                         "paper_id": r.paper_id, "page": r.page, "page_end": r.page_end,
                         "answerable": True, "source": "llm"})
            print("✅ Berhasil disimpan")
        else:
            print("❌ Dilewati (Skip/Halusinasi/Error)")

        # PERBAIKAN 3: Memberikan jeda waktu (delay) agar tidak terkena Rate Limit 15 RPM
        # 4 detik per request membuat kecepatan rata-rata Anda aman di ~15 request per menit.
        time.sleep(4.5)

    (BASE / "eval").mkdir(exist_ok=True)
    pd.DataFrame(rows).to_csv(BASE / "eval" / "candidates.csv", index=False)
    print(f"\nSelesai! Berhasil mengumpulkan {len(rows)} kandidat.")


Processing 1/100: Paper ID 2202.08341v1... ✅ Berhasil disimpan
Processing 2/100: Paper ID 2509.24448v1... ✅ Berhasil disimpan
Processing 3/100: Paper ID 2406.07176v2... Error generating response: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
❌ Dilewati (Skip/Halusinasi/Error)
Processing 4/100: Paper ID 2404.06564v4... ✅ Berhasil disimpan
Processing 5/100: Paper ID 2511.12909v1... ❌ Dilewati (Skip/Halusinasi/Error)
Processing 6/100: Paper ID 2603.13964v1... ✅ Berhasil disimpan
Processing 7/100: Paper ID 2104.13702v1... Error generating response: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
❌ Dilewati (Skip/Halusinasi/Error)
Processing 8/100: Paper ID 2505.05137v1... Error generating response: 

### 5.5 Load the candidates from Drive

Confirms the candidates were saved to Drive (66 rows).

**Unanswerable questions:** `eval/unanswerable_candidates.csv` holds 12 LLM-generated questions whose answers are not in the corpus. The cell that produced it is not part of this notebook; per the plan, the questions come from abstracts of search results that were left out of the corpus. These questions are not used for retrieval metrics; they test whether the answering step (M4–M5) refuses correctly.

In [25]:
import pandas as pd
from pathlib import Path

# Google Drive target folder
GD_BASE = Path.home() / "Projects" / "scholar-rag" / "data"
candidates_path = GD_BASE / "eval" / "candidates.csv"

print(f"Target path: {candidates_path}")
print(f"File exists: {candidates_path.exists()}")

if candidates_path.exists():
    df_candidates = pd.read_csv(candidates_path)
    print(f"Successfully loaded candidates. Total rows: {len(df_candidates)}")
    display(df_candidates.head())
else:
    print("File not found on Google Drive. Let's list the contents of the directory:")
    if GD_BASE.exists():
        print(list(GD_BASE.glob("**/*")))


Target path: /content/drive/MyDrive/scholar-rag/data/eval/candidates.csv
File exists: True
Successfully loaded candidates. Total rows: 66


,id,keep,question,answer,evidence,type,paper_id,page,page_end,answerable,source,skip
0,q001,NaN,What is the name of the open-source deep learning library introduced for unsupervised ...,Anomalib is an open-source deep learning library designed for unsupervised anomaly det...,"This paper introduces anomalib1, a novel library for unsupervised anomaly detection an...",factual,2202.08341v1,1,1,True,llm,NaN
1,q002,NaN,What is the name of the dual-model ensemble framework that uses two heterogeneous stud...,The proposed approach relies on a dual-model ensemble leveraging a shared pre-trained ...,"Our framework consists of a teacher and two student models: an Encoder-Decoder model, ...",method,2509.24448v1,1,1,True,llm,NaN
2,q003,NaN,How does MambaAD compare to DiAD in terms of parameter efficiency and performance acco...,MambaAD achieves approximately a 2.0 increase in performance while utilizing only one-...,our method MambaAD has achieved about 2.0 ↑improvement with only 1/50 the parameters a...,factual,2404.06564v4,9,10,True,llm,False
3,q004,NaN,Which specific pretrained language model is utilized for the independent text encoder ...,"The framework fine-tunes a pretrained BERT model, specifically 'bert-base-uncased', in...",We adopt a pretrained BERT model (bert-base-uncased) [22] and fine-tune it independent...,method,2603.13964v1,6,7,True,llm,NaN
4,q005,NaN,How does the performance of the DRÆM method compare to fully supervised approaches on ...,DRÆM surpasses all supervised methods in anomaly localization accuracy on the DAGM dat...,DRÆM outperforms all supervised methods in terms of anomaly localization accuracy on t...,factual,2108.07610v2,8,8,True,llm,NaN


---
## 6. Baseline retrieval (M2)

**Retrieval** is the first half of RAG: from ~3,900 chunks, pick the few most likely to contain the answer and pass only those to the LLM. If retrieval misses the right passage, no LLM can answer correctly, so retrieval is measured on its own first.

Two retrievers are compared. Both give every chunk a score for the question and rank chunks by that score; they differ in how the score is computed.

**Dense retrieval (meaning-based)**
- An embedding model turns every chunk into a vector (384 numbers for `bge-small-en-v1.5`), once, up front.
- The question is embedded the same way; chunks are ranked by cosine similarity to it.
- Strength: matches meaning even when words differ ("defect" ≈ "anomaly").
- Weakness: rare names and acronyms ("PatchCore", "SoftPatch+") are poorly represented once a long passage is squeezed into one vector.

**BM25 (keyword-based)**
- No model. Scores chunks by how many question words they contain, weighting rare words more ("PatchCore" counts more than "model"), with diminishing returns for repeats and a length correction.
- Strength: exact terms, names, numbers.
- Weakness: no notion of synonyms.

**Metrics.** Each question has one or two *gold chunks*: chunks from the gold paper that contain the evidence quote.

| Metric | Meaning |
|---|---|
| Hit@k | Share of questions whose gold chunk appears in the top *k* results |
| MRR | Mean of 1 / (rank of the first gold chunk); 0 if not found. Closer to 1 is better |
| PaperHit@10 | Share of questions whose gold *paper* appears in the top 10 (a looser measure) |

Hybrid search and reranking come in M3.

### 6.1 Setup and data

**Entry point for a fresh session:** mounts Drive and reloads chunks and question candidates, so sections 1–5 do not need to be re-run.

In [1]:
import json, re, numpy as np, pandas as pd
from pathlib import Path

BASE = Path.home() / "Projects" / "scholar-rag" / "data"
chunks = pd.read_json(BASE / "chunks.jsonl", lines=True, dtype={"paper_id": str})
cand_path = BASE / "eval" / "candidates.csv"
if not cand_path.exists():
    raise FileNotFoundError(
        "data/eval/candidates.csv belum ada. Salin dari Google Drive "
        "(MyDrive/scholar-rag/data/eval/) atau isi GEMINI_API_KEY lalu jalankan sel 5.4.")
qs = pd.read_csv(cand_path, dtype={"paper_id": str})
print(len(chunks), "chunk |", len(qs), "soal")

3991 chunk | 66 soal


### 6.2 Remove bibliography chunks and non-self-contained questions

- **Bibliography filter.** A chunk is treated as a reference list if it has at least 5 year mentions *and* at least 8 initials (`J.`, `K.`), or the phrase "arXiv preprint" twice. This removed **81 chunks**; the three printed samples are all reference lists. The remaining **3,910 chunks** are saved to `chunks_clean.jsonl` and form the search index.
- **Question filter.** Questions that say "this paper", "the authors", etc. cannot be answered without knowing which paper is meant. **2 such questions were dropped** (66 → 64).

In [2]:
def looks_like_refs(t):
    years = len(re.findall(r"\b(?:19|20)\d{2}[a-z]?\b", t))
    initials = len(re.findall(r"\b[A-Z]\.", t))
    return (years >= 5 and initials >= 8) or t.count("arXiv preprint") >= 2

chunks["is_ref"] = chunks.text.apply(looks_like_refs)
print(chunks.is_ref.sum(), "chunk terdeteksi sebagai daftar pustaka")
for t in chunks[chunks.is_ref].text.sample(3, random_state=0):
    print("-", t[:200], "...\n")

index_chunks = chunks[~chunks.is_ref].reset_index(drop=True)
index_chunks.to_json(BASE / "chunks_clean.jsonl", orient="records", lines=True, force_ascii=False)

# soal yang jelas tidak berdiri sendiri
bad_q = qs.question.str.contains(
    r"\b(?:this paper|the authors?|the passage|this work|the proposed method)\b",
    case=False, regex=True)
print(bad_q.sum(), "soal tidak berdiri sendiri, dibuang")
qs = qs[~bad_q].drop_duplicates("question").reset_index(drop=True)

81 chunk terdeteksi sebagai daftar pustaka
- 14902–14912. Schl¨uter, H. M.; Tan, J.; Hou, B.; and Kainz, B. 2022. Natural synthetic anomalies for self-supervised anomaly detection and localization. In Proceedings of European Conference on Comput ...

- Congress on Ultra Modern Telecommunications and Control Systems and Workshops (ICUMT), pages 66–71. IEEE, 2021. 7, 6 [25] Xi Jiang, Jianlin Liu, Jinbao Wang, Qiang Nie, Kai Wu, Yong Liu, Chengjie Wang ...

- 2019. Revisiting point cloud classification: A new benchmark dataset and classification model on real-world data. In ICCV, 1588–1597. Seoul, Korea. Vaswani, A.; Shazeer, N.; Parmar, N.; Uszkoreit, J.; ...

2 soal tidak berdiri sendiri, dibuang


### 6.3 Find the gold chunks

For each question, the gold chunks are the chunks of the gold paper whose text contains the evidence quote (after whitespace and case normalisation).

- 51 questions have 1 gold chunk, 13 have 2 (the quote falls inside the 60-word overlap of two neighbouring chunks).
- No question has 0 gold chunks, so all **64** questions are evaluated.

In [3]:
def norm(s): return re.sub(r"\s+", " ", s).strip().lower()

index_chunks["norm"] = index_chunks.text.map(norm)
gold = {}
for _, q in qs.iterrows():
    mask = (index_chunks.paper_id == q.paper_id) & \
           index_chunks.norm.str.contains(norm(q.evidence), regex=False)
    gold[q.id] = set(index_chunks.index[mask])

qs["n_gold"] = qs.id.map(lambda i: len(gold[i]))
print(qs.n_gold.value_counts().sort_index())      # 1-2 normal (overlap), 0 bermasalah
qs = qs[qs.n_gold > 0].reset_index(drop=True)
print(len(qs), "soal siap dievaluasi")

n_gold
1    51
2    13
Name: count, dtype: int64
64 soal siap dievaluasi


### 6.4 Dense index

- Model: `BAAI/bge-small-en-v1.5`, normalised embeddings, batch size 64.
- **Cache:** embeddings are saved to Drive together with the list of chunk ids. They are recomputed only if the chunk list changes.
- **Query prefix:** bge models expect short queries to start with "Represent this sentence for searching relevant passages: ".
- **Search:** a plain matrix product over 3,910 × 384 is exact and fast. A vector database (Chroma, FAISS) only becomes useful at deployment or much larger scale.

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
(BASE / "index").mkdir(exist_ok=True)
emb_path, ids_path = BASE / "index" / "emb_bge_small.npy", BASE / "index" / "chunk_ids.json"
cur_ids = index_chunks.chunk_id.tolist()

if emb_path.exists() and ids_path.exists() and json.load(open(ids_path)) == cur_ids:
    E = np.load(emb_path)
else:
    E = model.encode(index_chunks.text.tolist(), batch_size=64,
                     normalize_embeddings=True, show_progress_bar=True)
    np.save(emb_path, E); json.dump(cur_ids, open(ids_path, "w"))
print(E.shape)

QP = "Represent this sentence for searching relevant passages: "   # prefix query khusus model bge

def dense_rank(query, k):
    qv = model.encode([QP + query], normalize_embeddings=True)[0]
    return np.argsort(-(E @ qv))[:k].tolist()

W1019 15:43:21.967000 18327 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(3910, 384)


### 6.5 BM25 index

Text is lower-cased, split into alphanumeric tokens and stripped of English stop words before indexing.

In [5]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def tok(s): return [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in ENGLISH_STOP_WORDS]

bm25 = BM25Okapi([tok(t) for t in index_chunks.text])

def bm25_rank(query, k):
    return np.argsort(-bm25.get_scores(tok(query)))[:k].tolist()

### 6.6 Evaluate

For each question, both retrievers return their top 20 chunks; the metrics come from the rank of the first gold chunk.

**Reading the results**
- BM25 beats dense by a wide margin (Hit@1 0.625 vs 0.391). One question ≈ 1.6 points, so this gap is far beyond noise.
- BM25 finds the right **paper** in the top 10 for all 64 questions; in its 4 top-10 misses the paper is there, but not the gold passage.
- Dense misses the gold chunk in the top 10 for 13 questions, and for 5 of them it does not find the paper at all.
- Part of BM25's advantage comes from how the questions were made (they reuse passage wording), and part from dense truncation (section 6.9). This is not evidence that BM25 is better in general.

In [6]:
def evaluate(rank_fn, ks=(1, 5, 10, 20)):
    rows = []
    for _, q in qs.iterrows():
        ranked = rank_fn(q.question, max(ks))
        g = gold[q.id]
        r_chunk = next((i + 1 for i, c in enumerate(ranked) if c in g), np.nan)
        papers = index_chunks.paper_id.values[ranked]
        r_paper = next((i + 1 for i, p in enumerate(papers) if p == q.paper_id), np.nan)
        rows.append((q.id, r_chunk, r_paper))
    res = pd.DataFrame(rows, columns=["id", "rank_chunk", "rank_paper"])
    out = {f"Hit@{k}": (res.rank_chunk <= k).mean() for k in ks}
    out["MRR"] = (1 / res.rank_chunk).fillna(0).mean()
    out["PaperHit@10"] = (res.rank_paper <= 10).mean()
    return out, res

m_dense, res_dense = evaluate(dense_rank)
m_bm25,  res_bm25  = evaluate(bm25_rank)
print(pd.DataFrame({"Dense": m_dense, "BM25": m_bm25}).round(3))

             Dense   BM25
Hit@1        0.391  0.625
Hit@5        0.656  0.906
Hit@10       0.797  0.938
Hit@20       0.828  0.969
MRR          0.512  0.751
PaperHit@10  0.922  1.000


### 6.7 Inspect dense failures

Prints up to ten questions where the gold chunk is not in dense's top 10, with the gold paper and dense's top-1 paper. In several of them (e.g. q006, q007) the top-1 chunk comes from the **gold paper itself**: the right paper, a different passage. Such cases point to questions that more than one passage of the same paper can answer, rather than to a retrieval failure.

In [7]:
miss = res_dense[(res_dense.rank_chunk.isna()) | (res_dense.rank_chunk > 10)].id
qi = qs.set_index("id")
for i in miss.head(10):
    q = qi.loc[i]
    top1 = index_chunks.iloc[dense_rank(q.question, 1)[0]]
    print(f"\n[{i}] {q.question}")
    print("  jawaban :", q.answer)
    print("  paper emas :", index_chunks[index_chunks.paper_id == q.paper_id].title.iloc[0][:80])
    print("  top-1 dense:", top1.title[:80])


[q006] What two training strategies are combined in OmniAD to inject industrial knowledge and strengthen multimodal reasoning?
  jawaban : OmniAD combines Supervised Fine-Tuning (SFT) to teach response formats and object-specific knowledge with Group Relative Policy Optimization (GRPO) to refine the reasoning behavior using verifiable rewards.
  paper emas : OmniAD: Detect and Understand Industrial Anomaly via Multimodal Reasoning
  top-1 dense: OmniAD: Detect and Understand Industrial Anomaly via Multimodal Reasoning

[q007] What is the underlying architecture of the reconstruction network used in the Noise-to-Norm framework for industrial anomaly detection?
  jawaban : The reconstruction network in the Noise-to-Norm approach is built upon the M-net architecture, incorporating pixel shuffle operations and residual attention modules to enhance feature representation and reconstruction quality.
  paper emas : Noise-to-Norm Reconstruction for Industrial Anomaly Detection and Localizatio

### 6.8 Freeze the question set

Saves the 64 evaluated questions to `eval/questions_v1.csv`. From now on every experiment reads this file, so scores from different days stay comparable. Any later change is saved as `questions_v2.csv`, never over v1.

In [8]:
q_path = BASE / "eval" / "questions_v1.csv"
if q_path.exists():
    print("questions_v1.csv sudah ada (beku), tidak ditimpa.")
else:
    qs.drop(columns=["norm"], errors="ignore").to_csv(q_path, index=False)
    print(len(qs), "soal dibekukan di questions_v1.csv")


questions_v1.csv sudah ada (beku), tidak ditimpa.


### 6.9 Token-length check (dense truncation)

`bge-small-en-v1.5` reads at most 512 tokens and silently cuts the rest.

| Statistic | Tokens |
|---|---|
| Median chunk length | 471 |
| 75th percentile | 508 |
| Max | 925 |
| **Chunks over the limit** | **22.5%** |

Technical text costs about 1.45 tokens per word, so 350 words is not 350 tokens. The tokenizer warning in the output is harmless: lengths are counted here without truncation on purpose.

**Next experiment:** rebuild the chunks at 250 and 180 words and compare Hit@k and MRR for both retrievers. Because gold is defined by evidence quotes, the same frozen questions work for every chunk size. Smaller chunks also help the reranker in M3, which has the same 512-token limit for question and passage together.

In [9]:
tk = model.tokenizer
lens = pd.Series([len(tk(t, truncation=False)["input_ids"]) for t in index_chunks.text])
print("batas model:", model.max_seq_length)
print(lens.describe().round(0))
print("chunk yang terpotong:", round((lens > model.max_seq_length).mean(), 3))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (527 > 512). Running this sequence through the model will result in indexing errors


batas model: 512
count    3910.0
mean      468.0
std        75.0
min        87.0
25%       432.0
50%       471.0
75%       508.0
max       925.0
dtype: float64
chunk yang terpotong: 0.225


---
## 7. Evaluation set B: handwritten questions (in progress)

Why a second set: LLM-written questions reuse the passage's wording, which favours BM25. Handwritten questions are paraphrased the way a real user would ask, so they test whether retrieval understands meaning. Results are always reported **per source** (`llm` vs `handwritten`); if BM25's lead shrinks on handwritten questions, part of its advantage in section 6 came from question wording.

| Set | Size | Used for |
|---|---|---|
| A. LLM-generated | 64 | Retrieval metrics |
| B1. Handwritten, answerable | 15 | Retrieval metrics, reported separately |
| B2. Handwritten, unanswerable | 20 | Refusal testing in M4–M5 (not retrieval) |

**File format** (`eval/questions_handwritten_v1.csv`): `id` (`h001…` answerable, `hu001…` unanswerable), `question`, `answer`, `evidence` (exact 8–25-word quote), `type`, `paper_id`, `answerable`, `notes`. IDs must not collide with set A (`q…`), and evidence must be copied from chunk text, not from the PDF viewer, whose characters (ligatures, curly quotes) differ.

### 7.1 More robust text normalisation

Replaces `norm` with a version that applies Unicode NFKC (turns ligatures such as `ﬁ` into `fi`) and re-joins hyphenated line breaks, then recomputes the normalised chunk text. Needed because handwritten evidence is copied by hand.

In [10]:
import unicodedata, re
def norm(s):
    s = unicodedata.normalize("NFKC", str(s))          # ﬁ -> fi, dsb.
    s = re.sub(r"-\s+(\w)", r"\1", s)                  # sambung kata yang terpotong
    return re.sub(r"\s+", " ", s).strip().lower()

index_chunks["norm"] = index_chunks.text.map(norm)

### 7.2 Papers to read

Draws 20 random corpus papers that set A does not use, and prints their title and abstract. This spreads set B over a different part of the corpus.

In [11]:
sel = pd.read_json(BASE / "selected.jsonl", lines=True, dtype={"id": str})
sel["pid"] = sel["id"].str.replace("/", "_")
used = set(qs.paper_id)                                  # hindari paper yang sudah dipakai soal LLM
pick = sel[~sel.pid.isin(used)].sample(20, random_state=1)
for _, r in pick.iterrows():
    print(f"\n[{r.pid}] {r.title}\n{r.abstract[:600]} ...")


[2508.21795v1] TMUAD: Enhancing Logical Capabilities in Unified Anomaly Detection Models with a Text Memory Bank
Anomaly detection, which aims to identify anomalies deviating from normal patterns, is challenging due to the limited amount of normal data available. Unlike most existing unified methods that rely on carefully designed image feature extractors and memory banks to capture logical relationships between objects, we introduce a text memory bank to enhance the detection of logical anomalies. Specifically, we propose a Three-Memory framework for Unified structural and logical Anomaly Detection (TMUAD). First, we build a class-level text memory bank for logical anomaly detection by the proposed logi ...

[2507.21555v1] Multi-View Reconstruction with Global Context for 3D Anomaly Detection
3D anomaly detection is critical in industrial quality inspection. While existing methods achieve notable progress, their performance degrades in high-precision 3D anomaly detection due to insuf

### 7.3 Validate the handwritten questions

For each answerable question, checks that the evidence is found in the stated paper (`gold > 0`) and measures **overlap**: the share of the question's content words that also appear in the evidence. Good paraphrased questions stay below ~0.5.

**Current state:** 9 of 15 answerable questions find their evidence. The 6 failures and their causes:

| ID | Cause |
|---|---|
| h001 | The evidence cell holds a note instead of a quote |
| h007, h010, h012 | A trailing `...` was copied along with the quote |
| h013 | Copying typo ("100 images" instead of "100k") |
| h002 | Likely a curly apostrophe in the PDF (`’`) vs a straight one (`'`) in the quote |

Other pending fixes: five questions mention arXiv ids (h001, h005, h009, h011, h015) and should name the method instead; h002's type should be `factual`.

**Update 2026-10-19:** all six evidence quotes were re-copied from the chunk text and now match (15/15). h002 and h012 failed because the quote came from the arXiv abstract page, whose wording differs from the PDF version (not because of the apostrophe). h002's type is now `factual`.

**Validation rule (updated 2026-10-19):** an unanswerable question must name a corpus paper in `paper_id` (near-miss design, see 7.4) and have no evidence. The check also flags arXiv ids in a question and answerable `type` values outside `factual` / `method` / `limitation`.

In [12]:
hw_path = BASE / "eval" / "questions_handwritten_v1.csv"
if hw_path.exists():
    hw = pd.read_csv(hw_path, dtype=str).fillna("")
else:
    print("questions_handwritten_v1.csv belum ada (salin dari Drive); bagian ini memakai set kosong.")
    hw = pd.DataFrame(columns=["id", "question", "answer", "evidence", "paper_id", "answerable"])
hw["answerable"] = hw.answerable.str.upper().eq("TRUE")
print(len(hw), "soal |", hw.answerable.sum(), "terjawab |", (~hw.answerable).sum(), "tak-terjawab")

def gold_for(row):
    g = set()
    pids = [p.strip() for p in row.paper_id.split(";") if p.strip()]
    evs = [e.strip() for e in row.evidence.split(";") if e.strip()]
    for pid, ev in zip(pids, evs):
        m = (index_chunks.paper_id == pid) & index_chunks.norm.str.contains(norm(ev), regex=False)
        g |= set(index_chunks.index[m])
    return g

ARXIV_ID = re.compile(r"\b\d{4}\.\d{4,5}(v\d+)?\b")
ANS_TYPES = {"factual", "method", "limitation"}
corpus_papers = set(index_chunks.paper_id)

problems = []
for _, r in hw.iterrows():
    if ARXIV_ID.search(r.question):
        problems.append((r.id, "soal menyebut arXiv id; ganti dengan nama metode"))
    if r.answerable:
        if r.type not in ANS_TYPES:
            problems.append((r.id, f"type '{r.type}' bukan {sorted(ANS_TYPES)}"))
        g = gold_for(r)
        ev_toks = set(tok(r.evidence))
        ov = len(set(tok(r.question)) & ev_toks) / max(len(set(tok(r.question))), 1)
        n_ev = len(r.evidence.split())
        if not g:
            problems.append((r.id, "evidence tidak ditemukan di chunk paper itu"))
        if not 8 <= n_ev <= 40:
            problems.append((r.id, f"evidence {n_ev} kata"))
        print(f"{r.id}: gold={len(g)} | overlap kata soal-evidence={ov:.2f}")
    else:
        # desain near-miss: paper-nya ada di korpus, tapi detail yang ditanya tidak ada
        pid = r.paper_id.split(";")[0].strip()
        if not pid:
            problems.append((r.id, "soal tak-terjawab harus punya paper_id (desain near-miss)"))
        elif pid not in corpus_papers:
            problems.append((r.id, f"paper_id {pid} tidak ada di korpus"))
        if r.evidence:
            problems.append((r.id, "soal tak-terjawab tidak boleh punya evidence"))

print("\nMasalah:", problems or "tidak ada")

35 soal | 15 terjawab | 20 tak-terjawab
h001: gold=1 | overlap kata soal-evidence=0.21
h002: gold=1 | overlap kata soal-evidence=0.56
h003: gold=1 | overlap kata soal-evidence=0.40
h004: gold=1 | overlap kata soal-evidence=0.15
h005: gold=1 | overlap kata soal-evidence=0.25
h006: gold=1 | overlap kata soal-evidence=0.33
h007: gold=1 | overlap kata soal-evidence=0.50
h008: gold=1 | overlap kata soal-evidence=0.54
h009: gold=1 | overlap kata soal-evidence=0.36
h010: gold=1 | overlap kata soal-evidence=0.00
h011: gold=1 | overlap kata soal-evidence=0.23
h012: gold=1 | overlap kata soal-evidence=0.06
h013: gold=1 | overlap kata soal-evidence=0.11
h014: gold=1 | overlap kata soal-evidence=0.29
h015: gold=1 | overlap kata soal-evidence=0.00

Masalah: tidak ada


### 7.4 Are the unanswerable questions really unanswerable?

All 20 unanswerable questions point to papers that **are** in the corpus. They are *near-miss* questions: the paper exists, but the specific detail (GPU model, random seed, weight decay, …) is not in it. These are harder than out-of-corpus questions, because they test whether the model refuses instead of inventing a plausible detail.

Still to do before freezing:
1. Search each paper's chunks for the detail (keywords such as "weight decay", "seed", "GPU"). Several were judged from the abstract only, and implementation details often appear in the full text.
2. Name the method in the vague questions (hu010, hu014, hu016, hu020), so another paper cannot answer them.

In [13]:
corp = set(index_chunks.paper_id)
for _, r in hw[~hw.answerable].iterrows():
    pid = r.paper_id.split(";")[0].strip()
    print(f"{r.id} | paper_id: {pid or '-'} | ada di korpus: {pid in corp} | {r.question[:80]}")

hu001 | paper_id: 2508.21795v1 | ada di korpus: True | Which CUDA and PyTorch versions were used when measuring the inference time in T
hu002 | paper_id: 2507.21555v1 | ada di korpus: True | How does the inference latency of Multi-View Reconstruction (MVR) compare direct
hu003 | paper_id: 2408.11561v2 | ada di korpus: True | Which GPU and optimizer were used to train the model in the Iterative Refinement
hu004 | paper_id: 2511.06644v3 | ada di korpus: True | What is the total memory footprint in megabytes required by UniADC during zero-s
hu005 | paper_id: 2607.13499v1 | ada di korpus: True | What is the exact FLOPs count for the Memory-to-Prototype module when processing
hu006 | paper_id: 2111.09099v6 | ada di korpus: True | How does the proposed self-supervised predictive block perform on infrared therm
hu007 | paper_id: 2403.14233v1 | ada di korpus: True | What specific random seed was used to generate the artificial label-level noise 
hu008 | paper_id: 2408.02181v2 | ada di korpus: 

### 7.5 Combined evaluation, reported per source

Merges set A and the answerable part of set B, then reports Hit@1, Hit@10 and MRR per source and retriever.

**Result (2026-10-19, after the fixes in 7.3–7.4):** the `llm` row has n = 64 and reproduces the section 6 scores, and the `handwritten` row has n = 15. On handwritten questions dense wins Hit@1 (0.600 vs 0.200) and MRR (0.665 vs 0.442), while BM25 keeps a small lead at Hit@10 (0.933 vs 0.800). An earlier saved output (n = 18 and n = 67) was invalid: it predated the id rename, when ids like `q001` existed in both sets and overwrote each other's gold chunks.

In [14]:
qs_llm = pd.read_csv(BASE / "eval" / "questions_v1.csv", dtype={"paper_id": str})
qs_llm["source"] = "llm"
hw_ans = hw[hw.answerable].copy()
hw_ans["source"] = "handwritten"

qs = pd.concat([qs_llm, hw_ans], ignore_index=True)
gold = {}
for _, r in qs.iterrows():
    gold[r.id] = gold_for(r) if r.source == "handwritten" else set(
        index_chunks.index[(index_chunks.paper_id == r.paper_id) &
                           index_chunks.norm.str.contains(norm(r.evidence), regex=False)])
qs = qs[qs.id.map(lambda i: len(gold[i]) > 0)].reset_index(drop=True)
qs["paper_id"] = qs.paper_id.str.split(";").str[0]        # PaperHit memakai paper pertama

_, rd = evaluate(dense_rank)
_, rb = evaluate(bm25_rank)

def summarize(res):
    res = res.merge(qs[["id", "source"]], on="id")
    return res.groupby("source").apply(lambda d: pd.Series({
        "n": len(d),
        "Hit@1": (d.rank_chunk <= 1).mean(),
        "Hit@10": (d.rank_chunk <= 10).mean(),
        "MRR": (1 / d.rank_chunk).fillna(0).mean(),
    }))

print(pd.concat({"Dense": summarize(rd), "BM25": summarize(rb)}, axis=1).round(3))

            Dense                       BM25                     
                n  Hit@1 Hit@10    MRR     n  Hit@1 Hit@10    MRR
source                                                           
handwritten  15.0  0.600  0.800  0.665  15.0  0.200  0.933  0.442
llm          64.0  0.391  0.797  0.512  64.0  0.625  0.938  0.751


---
## 8. Next steps

1. ~~**Finish set B**~~ ✅ done 2026-10-19: evidence quotes fixed, arXiv ids replaced, unanswerable questions checked against the full text, validation rule updated; `questions_handwritten_v1.csv` is frozen.
2. ~~**Chunk-size experiment**~~ ✅ done (`02_chunk_size_experiment.ipynb`): keep 350 words; retest 250 with the reranker.
3. **M3 – Hybrid + rerank:** ✅ done (`03_hybrid_rerank.ipynb`): hybrid + `bge-reranker-base` ties BM25 on set A (Hit@1 0.656 vs 0.625, MRR 0.743 vs 0.752) and is best on set B1 (Hit@1 0.733). Final setup after the candidate-pool fix (sections 7–8 of that notebook): rerank BM25's top 10 ∪ dense's top 10 instead of RRF's top 20. It recovers three of the four questions RRF dropped, costs fewer pairs (≈ 16), ties BM25 on set A at Hit@5 and Hit@10, and keeps Hit@1 0.733 on set B1. Original plan: combine dense and BM25 rankings with Reciprocal Rank Fusion (score = Σ 1 / (60 + rank)), then rerank the top 20 with a cross-encoder. Target: beat BM25 on Hit@1 and MRR on set A without losing dense's lead on set B1. Test on 350- and 250-word chunks.
4. **M4 – Answers with citations:** ✅ done (`04_answers_citations.ipynb`): 63 / 64 set A and 14 / 15 set B1 answered (both refusals are retrieval misses), 20 / 20 set B2 refused, 76 / 77 citations verbatim. Original plan: the LLM answers in JSON (`answerable`, `answer`, `citations`); citations are checked in code against the chunks actually provided.
5. **M5 – Answer evaluation:** ✅ done (`05_evaluation.ipynb`): LLM judge on all 79 answers, acceptable 0.922 (A) and 0.933 (B1), no incorrect answers, no unsupported answer on unanswerable questions; judge-vs-human agreement 0.76 (the judge is stricter about completeness). Original plan: citation correctness, refusal accuracy on unanswerable questions, cost and latency per question.
6. **M6 – Ship it:** FastAPI endpoint, simple UI, public demo.